In [ ]:

from google.colab import drive
drive.mount('/content/drive')



In [ ]:
!pip install -q datasets sentence-transformers transformers faiss-cpu nltk gradio torch accelerate sacremoses


In [ ]:
!pip install -U bitsandbytes==0.46.1 accelerate transformers


In [ ]:
import pandas as pd

results_log = []


In [ ]:
import torch
import re
import nltk
import numpy as np
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Sentence tokenization for sentence-level hallucination detection
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
print(sent_tokenize("Chemotherapy treats cancer. It has side effects. It may cause nausea."))

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')


In [ ]:
# import os

# if os.path.exists("/content/drive/MyDrive/context_embeddings.npy"):
#     os.remove("/content/drive/MyDrive/context_embeddings.npy")

# if os.path.exists("/content/drive/MyDrive/faiss_index.bin"):
#     os.remove("/content/drive/MyDrive/faiss_index.bin")

# print("Old embeddings deleted.")

In [ ]:
# ================================
# 📚 PubMed Evidence Corpus Builder (Targeted + Improved)
# ================================

from datasets import load_dataset
import re
from nltk.tokenize import sent_tokenize
from tqdm import tqdm

print("Loading PubMedQA dataset...")

# Load PubMed QA dataset
dataset = load_dataset("pubmed_qa", "pqa_unlabeled", split="train")


def clean_text(text):
    """
    Clean biomedical text by removing HTML tags
    and extra whitespace.
    """
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 🔥 Focused biomedical keywords (side effects + treatment safety focus)
biomedical_keywords = [
    "chemotherapy",
    "adverse",
    "side effect",
    "toxicity",
    "complication",
    "nausea",
    "vomiting",
    "fatigue",
    "infection",
    "neutropenia",
    "hair loss",
    "diarrhea",
    "mortality",
    "survival",
    "risk",
    "associated with"
]

contexts = []

print("Extracting medical evidence sentences...")

for sample in tqdm(dataset):
    ctx_list = sample["context"]["contexts"]

    for ctx in ctx_list:
        cleaned = clean_text(ctx)
        sentences = sent_tokenize(cleaned)

        for sent in sentences:
            sent_lower = sent.lower()

            # Improved filtering logic
            if (
                len(sent.split()) > 8 and
                any(keyword in sent_lower for keyword in biomedical_keywords)
            ):
                contexts.append(sent.strip())

# Remove duplicates while preserving order
contexts = list(dict.fromkeys(contexts))

print("Total filtered medical evidence sentences:", len(contexts))

In [ ]:
print("Total contexts before limiting:", len(contexts))

MAX_CONTEXTS = 50000   # 🔥 Final decision
contexts = contexts[:MAX_CONTEXTS]

print("Using contexts:", len(contexts))


In [ ]:
print(contexts[0])

In [ ]:
# ==========================================
# 🔎 Embedding + FAISS Setup (Resumable + Cached)
# ==========================================

import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 🔥 Faster model for development (switch to large for final experiments)
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

EMBED_PATH = "/content/drive/MyDrive/context_embeddings.npy"
FAISS_PATH = "/content/drive/MyDrive/faiss_index.bin"

print("Loading embedding model...")
embedding_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=device
)

embedding_dim = embedding_model.get_sentence_embedding_dimension()

# ------------------------------------------------
# CASE 1: Both embeddings + FAISS exist → Load
# ------------------------------------------------
if os.path.exists(EMBED_PATH) and os.path.exists(FAISS_PATH):

    print("Loading saved embeddings and FAISS index...")
    context_embeddings = np.load(EMBED_PATH)
    index = faiss.read_index(FAISS_PATH)

# ------------------------------------------------
# CASE 2: Build embeddings (resumable)
# ------------------------------------------------
else:
    print("Building embeddings (resumable mode)...")

    BATCH_SIZE = 32

    # If partial embeddings exist, resume
    if os.path.exists(EMBED_PATH):
        print("Partial embeddings found. Resuming...")
        context_embeddings = np.load(EMBED_PATH)
        start_idx = context_embeddings.shape[0]
    else:
        context_embeddings = np.empty((0, embedding_dim), dtype="float32")
        start_idx = 0

    print(f"Resuming from index: {start_idx}")

    for i in range(start_idx, len(contexts), BATCH_SIZE):

        batch_contexts = contexts[i:i+BATCH_SIZE]

        batch_embeddings = embedding_model.encode(
            batch_contexts,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        context_embeddings = np.vstack((context_embeddings, batch_embeddings))

        # Save progress after each batch
        np.save(EMBED_PATH, context_embeddings)

        print(f"Saved embeddings up to index {min(i+BATCH_SIZE, len(contexts))}")

    print("Building FAISS index...")

    index = faiss.IndexFlatIP(embedding_dim)
    index.add(context_embeddings)

    faiss.write_index(index, FAISS_PATH)

    print("Embeddings and FAISS index saved successfully.")

# Mapping index → sentence
id_to_context = contexts

print("Index size:", index.ntotal)


# ==========================================
# 🔎 Retrieval Function
# ==========================================

def retrieve_evidence(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    retrieved_sentences = []
    retrieved_scores = []

    for i, score in zip(indices[0], scores[0]):
        retrieved_sentences.append(id_to_context[i])
        retrieved_scores.append(float(score))

    return retrieved_sentences, retrieved_scores


In [ ]:
test_query = "What are the side effects of chemotherapy?"

retrieved, scores = retrieve_evidence(test_query, top_k=3)

print("Scores:", scores)
print("Evidence:", retrieved)

In [ ]:
print(len(contexts))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
SAVE_PATH = "/content/drive/MyDrive/mistral_7b_model"

device = "cuda" if torch.cuda.is_available() else "cpu"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# ----------------------------------------
# Load from Drive if already saved
# ----------------------------------------
if os.path.exists(SAVE_PATH):

    print("Loading Mistral from Drive...")

    tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

    model = AutoModelForCausalLM.from_pretrained(
        SAVE_PATH,
        quantization_config=bnb_config,
        device_map="auto"
    )

else:
    print("Downloading Mistral (one-time)...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    # Save to Drive for future reuse
    tokenizer.save_pretrained(SAVE_PATH)
    model.save_pretrained(SAVE_PATH)

    print("Mistral saved to Drive.")

model.eval()

print("Mistral loaded successfully.")


In [ ]:
def generate_answer(query):

    prompt = f"""
You are a medical assistant.

Answer the following question in 5-6 clear and detailed sentences.
Be factual and medically accurate.

Question: {query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.3,
            top_p=0.9,
            do_sample=True
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt part
    answer = answer.split("Answer:")[-1].strip()

    return answer

In [ ]:
def retrieve_evidence(query, top_k=8, score_threshold=0.55):
    """
    Intent-Guided Dense Retrieval (IGDR)
    Retrieves clinically relevant biomedical evidence
    using semantic similarity + lightweight intent re-ranking.
    Uses global embedding_model, index, and id_to_context.
    """

    q_lower = query.lower()
    expanded_query = query

    # -------------------------
    # Intent-aware query expansion
    # -------------------------
    if any(k in q_lower for k in ["used for", "use", "indication", "benefit"]):
        expanded_query += " clinical use treatment indication"
    elif any(k in q_lower for k in ["side effect", "side effects", "risk", "adverse"]):
        expanded_query += " adverse effects clinical risk safety"
    elif any(k in q_lower for k in ["treat", "therapy", "management"]):
        expanded_query += " treatment therapy clinical management"

    # -------------------------
    # Encode query
    # -------------------------
    q_emb = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # -------------------------
    # Retrieve candidates
    # -------------------------
    scores, indices = index.search(q_emb, top_k * 3)

    evidence = []
    seen_texts = set()

    for j, i in enumerate(indices[0]):

        if i == -1:
            continue

        score = float(scores[0][j])
        text = id_to_context[i]

        if score < score_threshold or text in seen_texts:
            continue

        seen_texts.add(text)

        # -------------------------
        # Intent-aware re-ranking
        # -------------------------
        text_lower = text.lower()
        adjusted_score = score

        if "side effect" in q_lower and any(k in text_lower for k in ["side effect", "adverse", "risk"]):
            adjusted_score *= 1.1

        if "use" in q_lower and any(k in text_lower for k in ["used", "indicated", "treat"]):
            adjusted_score *= 1.1

        evidence.append({
            "text": text,
            "score": adjusted_score
        })

    # Sort by adjusted score
    evidence = sorted(evidence, key=lambda x: x["score"], reverse=True)

    return evidence[:top_k]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
torch_dtype = torch.float16 if device == "cuda" else torch.float32


LOAD THIS


In [ ]:
import os

SAVE_PATH = "/content/drive/MyDrive/mistral_7b"

print("Exists:", os.path.exists(SAVE_PATH))
if os.path.exists(SAVE_PATH):
    print("Files:", os.listdir(SAVE_PATH))


In [ ]:
import re

def filter_evidence_by_question(evidence, question, min_overlap=2):
    """
    Hybrid lexical filtering layer.
    Retains retrieved evidence that shares meaningful
    lexical tokens with the question.
    Falls back to original evidence if filtering removes everything.
    """

    STOPWORDS = {
        "what", "is", "are", "the", "a", "an", "for",
        "of", "to", "does", "do", "can", "be", "with",
        "how", "which", "about", "this", "that", "these",
        "those", "there", "their"
    }

    # Tokenize question
    q_words = {
        w for w in re.findall(r"\b\w+\b", question.lower())
        if w not in STOPWORDS and len(w) > 2
    }

    filtered = []

    for ev in evidence:
        ev_words = {
            w for w in re.findall(r"\b\w+\b", ev["text"].lower())
            if len(w) > 2
        }

        overlap = q_words & ev_words

        if len(overlap) >= min_overlap:
            filtered.append(ev)

    # 🔥 SAFETY FALLBACK
    if len(filtered) == 0:
        return evidence  # fallback to original retrieved evidence

    return filtered

In [ ]:
import torch

def generate_initial_answer(question, deterministic=True):
    """
    Baseline medical answer generation without retrieval.
    Used for hallucination comparison.
    Produces 4–6 sentence structured response.
    """

    prompt = (
        "You are a medically accurate assistant.\n"
        "Provide a detailed and factual answer in 5-6 clear sentences.\n"
        "Do not exaggerate or make unsupported claims.\n\n"
        f"Question: {question}\n\nAnswer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    generation_args = {
        "max_new_tokens": 220,
        "pad_token_id": tokenizer.eos_token_id,
        "repetition_penalty": 1.1
    }

    if deterministic:
        generation_args["do_sample"] = False
    else:
        generation_args.update({
            "do_sample": True,
            "temperature": 0.5,
            "top_p": 0.9
        })

    with torch.no_grad():
        output = model.generate(**inputs, **generation_args)

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Answer:" in text:
        text = text.split("Answer:", 1)[-1]

    return text.strip()

In [ ]:
import numpy as np
from nltk.tokenize import sent_tokenize

def detect_hallucination(answer, evidence, embedding_model, threshold=0.65):
    """
    Improved sentence-level hallucination detection.
    Uses cosine similarity with adaptive stability.
    """

    def is_valid_sentence(s):
        return (
            len(s.split()) >= 5 and
            not any(x in s for x in ["<", ">", "/", "[", "]"])
        )

    sentences = [s.strip() for s in sent_tokenize(answer) if is_valid_sentence(s)]

    if len(sentences) == 0:
        return {
            "is_hallucinated": True,
            "analysis": [],
            "hallucination_ratio": 1.0,
            "note": "No valid sentences generated."
        }

    if len(evidence) == 0:
        return {
            "is_hallucinated": None,
            "analysis": [],
            "hallucination_ratio": None,
            "note": "No supporting evidence retrieved."
        }

    # Slight relaxation for short answers
    if len(sentences) <= 2:
        threshold -= 0.05

    evidence_texts = [ev["text"] for ev in evidence]

    evidence_embs = embedding_model.encode(
        evidence_texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    sent_embs = embedding_model.encode(
        sentences,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    GENERIC_PHRASES = [
        "commonly used", "widely used", "generally safe",
        "it is known", "studies suggest", "may be used"
    ]

    results = []

    for i, sent in enumerate(sentences):

        sims = np.dot(evidence_embs, sent_embs[i])
        max_sim = float(np.max(sims))

        # Softer penalty
        generic_penalty = 0.02 if any(p in sent.lower() for p in GENERIC_PHRASES) else 0.0
        adjusted_sim = max_sim - generic_penalty

        results.append({
            "sentence": sent,
            "supported": adjusted_sim >= threshold,
            "similarity": round(adjusted_sim, 3)
        })

    hallucinated_count = sum(not r["supported"] for r in results)

    return {
        "is_hallucinated": hallucinated_count > 0,
        "analysis": results,
        "hallucination_ratio": round(hallucinated_count / len(results), 2)
    }

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

CORRECTOR_MODEL = "google/flan-t5-base"

print("Loading correction model (FLAN-T5)...")

corrector_tokenizer = AutoTokenizer.from_pretrained(CORRECTOR_MODEL)

corrector_model = AutoModelForSeq2SeqLM.from_pretrained(
    CORRECTOR_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

corrector_model = corrector_model.to(device)
corrector_model.eval()

print("Correction model loaded successfully.")


In [ ]:
import torch
import re
from nltk.tokenize import sent_tokenize

def self_correct(question, answer, evidence, report):

    # Use SAME sentence splitting logic as detection
    sentences = [s.strip() for s in sent_tokenize(answer) if len(s.split()) >= 5]

    bad_indices = [
        i for i, r in enumerate(report["analysis"])
        if not r["supported"]
    ]

    if not bad_indices:
        return answer

    bad_sentences = [sentences[i] for i in bad_indices]

    evidence_text = "\n".join(
        [f"- {ev['text'][:180]}" for ev in evidence[:5]]
    )

    bad_sent_text = "\n".join([f"- {s}" for s in bad_sentences])

    prompt = (
        "You are a medical fact-checking assistant.\n"
        "Rewrite ONLY the hallucinated sentences.\n"
        "Use only clear, simple, evidence-supported facts.\n"
        "Do not copy research abstracts.\n"
        "Keep sentences short and patient-friendly.\n\n"
        f"Question:\n{question}\n\n"
        f"Hallucinated sentences:\n{bad_sent_text}\n\n"
        f"Evidence:\n{evidence_text}\n\n"
        "Corrected sentences:"
    )

    inputs = corrector_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output = corrector_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.15
        )

    corrected_text = corrector_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    if "Corrected sentences:" in corrected_text:
        corrected_text = corrected_text.split("Corrected sentences:", 1)[-1]

    corrected_text = corrected_text.strip()

    # Clean unwanted biomedical jargon
    corrected_text = re.sub(
        r"\b(tumou?r|sample|measured|lipidperoxidation|GSH-Px)\b",
        "",
        corrected_text,
        flags=re.IGNORECASE
    )

    corrected_text = re.sub(r"\s{2,}", " ", corrected_text)

    corrected_sentences = sent_tokenize(corrected_text)

    # 🔥 Safety: if FLAN returns fewer sentences than expected
    if len(corrected_sentences) < len(bad_indices):
        corrected_sentences = corrected_sentences + bad_sentences[len(corrected_sentences):]

    new_answer = sentences.copy()

    for idx, new_sent in zip(bad_indices, corrected_sentences):
        new_answer[idx] = new_sent.strip()

    return " ".join(new_answer).strip()

In [ ]:
def medical_qa_pipeline(question, max_iters=2):

    global results_log

    MIN_HALLUCINATION_TRIGGER = 0.15
    MIN_IMPROVEMENT = 0.02

    if not question or not question.strip():
        return {
            "question": question,
            "initial": "",
            "final": "",
            "hallucination_ratio_before": 0.0,
            "hallucination_ratio_after": 0.0,
            "iterations_used": 0,
            "note": "Empty question"
        }

    # 1️⃣ Initial generation
    initial = generate_initial_answer(question)

    # 2️⃣ Retrieval (UPDATED CALL)
    evidence = retrieve_evidence(question, top_k=8)

    # 🔥 Apply lexical filtering
    evidence = filter_evidence_by_question(evidence, question)

    if not evidence:
        results_log.append({
            "question": question,
            "hallucination_before": 1.0,
            "hallucination_after": 1.0,
            "improvement": 0.0,
            "iterations": 0
        })

        return {
            "question": question,
            "initial": initial,
            "final": initial,
            "hallucination_ratio_before": 1.0,
            "hallucination_ratio_after": 1.0,
            "iterations_used": 0,
            "note": "Insufficient biomedical evidence retrieved"
        }

    # 3️⃣ Detect before correction
    detection_before = detect_hallucination(
        initial,
        evidence,
        embedding_model,
        threshold=0.65
    )

    before_ratio = detection_before["hallucination_ratio"]

    current_answer = initial
    current_detection = detection_before
    iteration_count = 0

    # 4️⃣ Correct only if meaningful hallucination
    if before_ratio >= MIN_HALLUCINATION_TRIGGER:

        for _ in range(max_iters):

            corrected = self_correct(
                question,
                current_answer,
                evidence,
                current_detection
            )

            new_detection = detect_hallucination(
                corrected,
                evidence,
                embedding_model,
                threshold=0.65
            )

            improvement = (
                current_detection["hallucination_ratio"]
                - new_detection["hallucination_ratio"]
            )

            if improvement >= MIN_IMPROVEMENT:
                current_answer = corrected
                current_detection = new_detection
                iteration_count += 1
            else:
                break

    after_ratio = current_detection["hallucination_ratio"]
    improvement_final = before_ratio - after_ratio

    # 5️⃣ Log results
    results_log.append({
        "question": question,
        "hallucination_before": round(before_ratio, 3),
        "hallucination_after": round(after_ratio, 3),
        "improvement": round(improvement_final, 3),
        "iterations": iteration_count
    })

    return {
        "question": question,
        "initial": initial,
        "final": current_answer,
        "hallucination_ratio_before": before_ratio,
        "hallucination_ratio_after": after_ratio,
        "iterations_used": iteration_count,
        "detection_before": detection_before,
        "detection_after": current_detection
    }

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')


In [ ]:
import html
from nltk.tokenize import sent_tokenize

def highlight_hallucinated_sentences(answer, detection_report):
    """
    Highlights hallucinated sentences in red.
    Uses index-based matching for stability.
    """

    if not detection_report or "analysis" not in detection_report:
        return html.escape(answer)

    # Use SAME filtering logic as detection
    original_sentences = sent_tokenize(answer)

    analysis = detection_report.get("analysis", [])

    highlighted_sentences = []

    for i, s in enumerate(original_sentences):

        escaped_s = html.escape(s.strip())

        # Safety check: avoid index mismatch
        if i < len(analysis) and not analysis[i].get("supported", True):
            highlighted_sentences.append(
                f"<span style='color:red; font-weight:bold'>{escaped_s}</span>"
            )
        else:
            highlighted_sentences.append(escaped_s)

    return " ".join(highlighted_sentences)

In [ ]:
import re
import html

def sanitize_for_ui(text):
    """
    Cleans model outputs for safe and readable UI display.
    Removes instruction artifacts without over-deleting content.
    """

    if not isinstance(text, str) or not text.strip():
        return ""

    # Remove special tokens
    text = re.sub(r"</?s>", "", text)
    text = re.sub(r"\[/?INST\]", "", text)

    # Remove leading 'Answer:' only if at beginning
    text = re.sub(r"^\s*Answer:\s*", "", text, flags=re.IGNORECASE)

    # Remove leading 'Corrected sentences:' safely
    text = re.sub(r"^\s*Corrected sentences:\s*", "", text, flags=re.IGNORECASE)

    # Normalize whitespace
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()

In [ ]:
def demo(question):
    try:
        if not question or not question.strip():
            return "⚠️ Please enter a medical question."

        result = medical_qa_pipeline(question)

        # Clean question
        clean_question = sanitize_for_ui(result["question"])

        if "detection_before" in result:

            # Sanitize initial BEFORE highlighting
            clean_initial = sanitize_for_ui(result["initial"])

            highlighted_initial = highlight_hallucinated_sentences(
                clean_initial,
                result["detection_before"]
            )

            correction_status = (
                "✅ Correction applied"
                if result["hallucination_ratio_after"] is not None and
                   result["hallucination_ratio_before"] is not None and
                   result["hallucination_ratio_after"] < result["hallucination_ratio_before"]
                else "ℹ️ No correction needed"
            )

            analysis_block = f"""
### 🔍 Hallucination Analysis
- **Before correction:** {result["hallucination_ratio_before"]:.2f}
- **After correction:** {result["hallucination_ratio_after"]:.2f}

**Status:** {correction_status}
"""

        else:
            highlighted_initial = sanitize_for_ui(result["initial"])
            analysis_block = """
### 🔍 Hallucination Analysis
⚠️ No sufficient biomedical evidence retrieved for verification.
"""

        # Sanitize final answer
        final_raw = sanitize_for_ui(result["final"])

        # 🔥 Important: Wrap highlighted block in div to ensure HTML rendering
        md = f"""
## 🩺 Explainable Self-Correcting Medical LLM

**Question:** {clean_question}

---

### 🤖 Initial Answer
🟥 *Hallucinated sentences highlighted (if any)*

<div>
{highlighted_initial}
</div>

---

### ✅ Final Answer (Evidence-Grounded)

{final_raw}

---

{analysis_block}
"""

        return md

    except Exception as e:
        return f"❌ Runtime Error:\n```\n{str(e)}\n```"

In [ ]:
import gradio as gr
import nltk

# Ensure sentence tokenizer available
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

with gr.Blocks() as demo_ui:
    gr.Markdown("## 🩺 Explainable Self-Correcting Medical LLM")
    gr.Markdown("**M.Tech Main Project | Research & Educational Use**")

    question = gr.Textbox(
        label="Ask a medical question",
        placeholder="e.g., Can antibiotics cure viral infections like flu?",
        lines=2
    )

    # 🔥 Allow HTML rendering for red highlights
    output = gr.Markdown()

    with gr.Row():
        submit = gr.Button("Submit")
        clear = gr.Button("Clear")

    submit.click(fn=demo, inputs=question, outputs=output)

    clear.click(
        fn=lambda: ["", ""],
        inputs=[],
        outputs=[question, output]
    )

    gr.Examples(
        examples=[
            "Can antibiotics cure viral infections like flu?",
            "What is metformin used for?",
            "Side effects of chemotherapy?",
            "Is long-term steroid use harmful?"
        ],
        inputs=question
    )

demo_ui.queue().launch(share=True, debug=False)

In [ ]:
# import pandas as pd

# df_results = pd.DataFrame(results_log)

# df_results


In [ ]:
import pandas as pd

df_results = pd.DataFrame(results_log)

df_results


In [ ]:
df_results = pd.DataFrame(results_log)

df_results


In [ ]:
df_results["reduction"] = (
    df_results["hallucination_before"] - df_results["hallucination_after"]
)

df_results


In [ ]:
print("Average hallucination before:", df_results["hallucination_before"].mean())
print("Average hallucination after :", df_results["hallucination_after"].mean())


In [ ]:
df_results["improvement"].value_counts()
